In [ ]:
validate_dataframe(customers, "Customers")
validate_dataframe(orders, "Orders")
validate_dataframe(sessions, "Sessions")

In [ ]:
customers = pd.read_csv("../data/processed/customers.csv")
orders = pd.read_csv("../data/processed/orders.csv")
sessions = pd.read_csv("../data/processed/sessions.csv")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../"))

from src.validation_engine import validate_dataframe
import pandas as pd

In [ ]:
# Conversion Funnel Analysis

This notebook analyzes the customer conversion funnel to identify
drop-off points and optimize the user journey from browsing to purchase.

Key goals:

• Measure conversion rates at each stage
• Identify bottlenecks in the purchase process
• Quantify the impact of cart abandonment

## Data Sources

This analysis uses the following datasets:

• customers.csv – customer acquisition and experiment groups
• sessions.csv – user browsing sessions
• cart_events.csv – cart interactions
• orders.csv – purchase transactions
• products.csv – product catalog

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

In [ ]:
sessions = pd.read_csv("../data/raw/sessions.csv")
orders = pd.read_csv("../data/raw/orders.csv")
customers = pd.read_csv("../data/raw/customers.csv")
products = pd.read_csv("../data/raw/products.csv")
cart = pd.read_csv("../data/raw/cart_events.csv")

In [ ]:
sessions_count = sessions.session_id.nunique()
cart_count = cart.session_id.nunique()
orders_count = orders.session_id.nunique()

funnel = pd.DataFrame({

"stage": ["Sessions","Cart","Orders"],

"count": [sessions_count,cart_count,orders_count]

})

sns.barplot(data=funnel,x="stage",y="count")

plt.title("Customer Funnel")

plt.show()

A significant drop occurs between cart interactions and completed
purchases, suggesting potential friction in the checkout process.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Funnel Metrics (Descriptive) - Order Conversion by Group

In [15]:
import pandas as pd

sessions = pd.read_csv("../data/raw/sessions.csv")
orders = pd.read_csv("../data/raw/orders.csv")

# Merge session-level AB group
session_ab = sessions.merge(
    pd.read_csv("../data/raw/customers.csv")[["customer_id", "ab_group"]],
    on="customer_id",
    how="left"
)

# Flag sessions that converted to at least one order
converted_sessions = orders["session_id"].unique()

session_ab["converted"] = session_ab["session_id"].isin(converted_sessions)

session_conversion = session_ab.groupby("ab_group")["converted"].mean()
session_conversion

ab_group
Control      0.174986
Treatment    0.183918
Name: converted, dtype: float64

# Funnel Metrics (Descriptive) - AOV Comparison

In [16]:
orders.groupby("ab_group")["final_price"].mean()

ab_group
Control      18151.014732
Treatment    11846.075292
Name: final_price, dtype: float64

# Funnel Metrics (Descriptive) - Margin Comparison

In [17]:
orders.groupby("ab_group")["margin"].mean()

ab_group
Control      2355.614872
Treatment   -3252.085102
Name: margin, dtype: float64

# Statistical Testing (Inferential) - Proportion Z-Test (Conversion)

In [18]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Prepare counts
session_ab = sessions.merge(
    customers[["customer_id", "ab_group"]],
    on="customer_id",
    how="left"
)

converted_sessions = orders["session_id"].unique()
session_ab["converted"] = session_ab["session_id"].isin(converted_sessions)

conversion_counts = session_ab.groupby("ab_group")["converted"].sum()
total_sessions = session_ab.groupby("ab_group")["converted"].count()

count = np.array(conversion_counts)
nobs = np.array(total_sessions)

z_stat, p_value = proportions_ztest(count, nobs)
z_stat, p_value

(np.float64(-1.2501100481414422), np.float64(0.2112593497138846))

# Proportion Z-Test (Conversion) - T-Test (AOV)

In [19]:
from scipy.stats import ttest_ind

control_orders = orders[orders["ab_group"] == "Control"]["final_price"]
treatment_orders = orders[orders["ab_group"] == "Treatment"]["final_price"]

ttest_ind(control_orders, treatment_orders, equal_var=False)

TtestResult(statistic=np.float64(7.991276896847967), pvalue=np.float64(1.8149357094268565e-15), df=np.float64(3384.7792030207056))

# Statistical Testing (Inferential) - T-Test (Margin)

In [20]:
control_margin = orders[orders["ab_group"] == "Control"]["margin"]
treatment_margin = orders[orders["ab_group"] == "Treatment"]["margin"]

ttest_ind(control_margin, treatment_margin, equal_var=False)

TtestResult(statistic=np.float64(29.88947231539061), pvalue=np.float64(4.570527848188216e-175), df=np.float64(3506.9908267590918))

While the discount treatment shows a nominal +0.89pp lift in conversion, the effect is not statistically significant (p=0.21). However, the treatment significantly reduces AOV and drives negative contribution margins per order (p < 0.001). Based on current economics, the discount strategy is not profit-accretive and should not be rolled out without optimization.

In [21]:
import pandas as pd
import numpy as np

from src.order_engine import OrderEngine

sessions = pd.read_csv("../data/raw/sessions.csv")
cart = pd.read_csv("../data/raw/cart_events.csv")
customers = pd.read_csv("../data/raw/customers.csv")
latent = pd.read_csv("../data/processed/customer_latent_traits.csv")
products = pd.read_csv("../data/raw/products.csv")

In [22]:
scenarios = [
    (0.10, 0.20),
    (0.15, 0.30),
    (0.20, 0.40),
    (0.25, 0.50)
]

results = []

for low, high in scenarios:

    engine = OrderEngine(
        cart,
        customers,
        latent,
        products,
        treatment_discount_range=(low, high)
    )

    sim_orders = engine.generate_orders()

    conversion = sim_orders["session_id"].nunique() / len(sessions)
    total_revenue = sim_orders["final_price"].sum()
    total_margin = sim_orders["margin"].sum()

    results.append({
        "discount_range": f"{int(low*100)}-{int(high*100)}%",
        "conversion_rate": round(conversion, 4),
        "total_revenue": round(total_revenue, 2),
        "total_margin": round(total_margin, 2)
    })

pd.DataFrame(results)

,discount_range,conversion_rate,total_revenue,total_margin
0,10-20%,0.1813,82015441.76,6396541.72
1,15-30%,0.1819,78208598.22,2202856.77
2,20-40%,0.1803,74257316.61,-1251593.74
3,25-50%,0.1827,68837236.60,-4754084.62


1️⃣ Conversion barely changes.

All bands:
~18.0–18.3%

That means:
Discounts are NOT materially improving conversion.
This is powerful insight.

2️⃣ Revenue decreases as discount increases.

Higher discounts → lower realized revenue.


3️⃣ Margin collapses aggressively after 20%.

This is the key inflection point.

10–20% → strong positive profit
15–30% → positive but weakened
20–40% → negative
25–50% → heavily negative

👉 Break-even threshold ≈ ~18–22%

Scenario simulation indicates minimal elasticity impact on conversion across discount bands. However, margin deteriorates sharply beyond 20% discount. Optimal band for maximizing total contribution margin lies within 10–20%.

# Build Session-Level Dataset

In [ ]:
import pandas as pd

# Reload data
sessions = pd.read_csv("../data/raw/sessions.csv")
orders = pd.read_csv("../data/raw/orders.csv")
customers = pd.read_csv("../data/raw/customers.csv")
latent = pd.read_csv("../data/processed/customer_latent_traits.csv")

# Merge AB + latent info
session_df = (
    sessions
    .merge(customers[["customer_id", "ab_group", "acquisition_channel"]],
           on="customer_id",
           how="left")
    .merge(latent,
           on="customer_id",
           how="left")
)

# Create target variable
converted_sessions = orders["session_id"].unique()
session_df["converted"] = session_df["session_id"].isin(converted_sessions).astype(int)

session_df.head()

,session_id,customer_id,session_date,device_type,traffic_source,pages_viewed,session_duration,bounced_flag,ab_group,acquisition_channel,activity_propensity,price_sensitivity,engagement_level,converted
0,1,1,2025-10-21,Mobile,Referral,6,340,0,Treatment,Referral,0.472577,0.412077,0.788016,0
1,2,1,2025-12-30,Mobile,Referral,2,401,0,Treatment,Referral,0.472577,0.412077,0.788016,0
2,3,1,2025-09-26,Mobile,Referral,2,124,0,Treatment,Referral,0.472577,0.412077,0.788016,0
3,4,1,2025-11-15,Mobile,Referral,4,280,0,Treatment,Referral,0.472577,0.412077,0.788016,1
4,5,1,2025-08-07,Mobile,Referral,3,320,0,Treatment,Referral,0.472577,0.412077,0.788016,0


# Feature Engineering and Logistic Regression Model

In [ ]:
import pandas as pd
import numpy as np

# Rebuild session_df cleanly
sessions = pd.read_csv("../data/raw/sessions.csv")
orders = pd.read_csv("../data/raw/orders.csv")
customers = pd.read_csv("../data/raw/customers.csv")
latent = pd.read_csv("../data/processed/customer_latent_traits.csv")

session_df = (
    sessions
    .merge(customers[["customer_id", "ab_group", "acquisition_channel"]],
           on="customer_id", how="left")
    .merge(latent, on="customer_id", how="left")
)

converted_sessions = orders["session_id"].unique()
session_df["converted"] = session_df["session_id"].isin(converted_sessions).astype(int)

# -----------------------------
# Feature Engineering
# -----------------------------

# Create weekend flag
session_df["session_date"] = pd.to_datetime(session_df["session_date"])
session_df["is_weekend"] = session_df["session_date"].dt.weekday >= 5

# Drop non-model columns
session_df = session_df.drop(columns=["session_id", "customer_id", "session_date"])

# Encode categoricals properly
categorical_cols = ["ab_group", "acquisition_channel", "device_type", "traffic_source"]

session_df = pd.get_dummies(
    session_df,
    columns=categorical_cols,
    drop_first=True
)

# Ensure weekend is numeric
session_df["is_weekend"] = session_df["is_weekend"].astype(int)

# -----------------------------
# Modeling
# -----------------------------

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X = session_df.drop(columns=["converted"])
y = session_df["converted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

AUC: 0.5895702178631578
              precision    recall  f1-score   support

           0       0.86      0.39      0.53      2847
           1       0.20      0.72      0.32       624

    accuracy                           0.45      3471
   macro avg       0.53      0.55      0.43      3471
weighted avg       0.74      0.45      0.50      3471

AUC: 0.5895702178631578
              precision    recall  f1-score   support

           0       0.86      0.39      0.53      2847
           1       0.20      0.72      0.32       624

    accuracy                           0.45      3471
   macro avg       0.53      0.55      0.43      3471
weighted avg       0.74      0.45      0.50      3471



# Interpret Coefficients

In [29]:
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0])
}).sort_values(by="odds_ratio", ascending=False)

coefficients.head(10)

,feature,coefficient,odds_ratio
5,engagement_level,0.378465,1.460042
10,acquisition_channel_Referral,0.189679,1.208862
15,traffic_source_Referral,0.189679,1.208862
13,traffic_source_Organic,0.168940,1.184049
8,acquisition_channel_Organic,0.168940,1.184049
14,traffic_source_Paid Ads,0.090099,1.094283
9,acquisition_channel_Paid Ads,0.090099,1.094283
3,activity_propensity,0.077571,1.080659
7,ab_group_Treatment,0.065564,1.067761
0,pages_viewed,0.048174,1.049353


I built a logistic regression model to quantify drivers of session conversion. Engagement and channel quality had the strongest impact, while discount treatment had only a marginal effect. This aligned with our statistical testing, suggesting behavioral drivers outweigh promotional incentives.